In [ ]:
def get_shakespeare_data():
    """Downloads Tiny Shakespeare dataset if not present and returns the text."""
    file_path = "input_.txt"
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    if not os.path.exists(file_path):
        print(f"Downloading dataset from {url}...")
        with open(file_path, "w") as f:
            f.write(requests.get(url).text)
        print("Download complete.")
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    return text[:19400]
    


In [ ]:
# Full end-to-end script (paste into a fresh notebook cell)
# Requirements: torch, wandb, requests
# pip install wandb   (uncomment and run if you need)

import os, time, requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wandb

# -------------------------
# DATA: download + tokenizer
# -------------------------


class CharTokenizer:
    def __init__(self, text):
        self.chars = sorted(list(set(text)))
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for i, ch in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
    def encode(self, s): return [self.stoi[c] for c in s]
    def decode(self, idxs): return ''.join([self.itos[i] for i in idxs])

class CharDataset(Dataset):
    def __init__(self, ids, block_size):
        self.ids = ids
        self.block_size = block_size
    def __len__(self): return len(self.ids) - self.block_size
    def __getitem__(self, i):
        x = torch.tensor(self.ids[i:i+self.block_size], dtype=torch.long)
        y = torch.tensor(self.ids[i+1:i+self.block_size+1], dtype=torch.long)
        return x, y

# -------------------------
# EXACT compressor / float embed / upsampler
# -------------------------
class ChunkCompressorEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, chunk_size, quantized_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim, padding_idx=0)
        self.conv = nn.Conv1d(in_channels=embedding_dim, out_channels=quantized_dim, kernel_size=chunk_size)
        self.proj = nn.Linear(quantized_dim, quantized_dim)
        nn.init.normal_(self.embedding.weight, std=0.02)
        nn.init.xavier_uniform_(self.conv.weight)
        nn.init.zeros_(self.conv.bias)
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)
    def forward(self, x):
        # x: (B*num_chunks, chunk_size)
        emb = self.embedding(x)           # (B*num_chunks, chunk_size, emb_dim)
        emb = emb.permute(0, 2, 1)        # (B*num_chunks, emb_dim, chunk_size)
        conv_out = self.conv(emb)         # (B*num_chunks, quantized_dim, 1)
        conv_out = conv_out.squeeze(-1)   # (B*num_chunks, quantized_dim)
        return self.proj(conv_out)        # (B*num_chunks, quantized_dim)

class InterpolatedFloatEmbedding(nn.Module):
    def __init__(self, input_dim, embedding_dim):
        super().__init__()
        self.embedding_matrix = nn.Parameter(torch.randn(input_dim, embedding_dim) * 0.01)
        self.ln = nn.LayerNorm(embedding_dim)
        self.input_dim = input_dim
        nn.init.normal_(self.embedding_matrix, std=0.01)
    def forward(self, x):
        # x: (B, N) floats in [0, input_dim-1]
        x_clamped = x.clamp(0.0, float(self.input_dim - 1))
        x_floor = torch.floor(x_clamped).long()
        x_ceil  = torch.ceil(x_clamped).long()
        w_high  = (x_clamped - x_floor).unsqueeze(-1)
        w_low   = 1.0 - w_high
        emb_floor = self.embedding_matrix[x_floor]
        emb_ceil  = self.embedding_matrix[x_ceil]
        return self.ln(w_low * emb_floor + w_high * emb_ceil)

class ChunkReconstructorMLP(nn.Module):
    def __init__(self, input_dim, chunk_size, vocab_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim * 2),
            nn.GELU(),
            nn.Linear(input_dim * 2, chunk_size * vocab_size)
        )
        self.chunk_size = chunk_size
        self.vocab_size = vocab_size
    def forward(self, chunk_embs):
        B, N, D = chunk_embs.shape
        out = self.net(chunk_embs)  # (B, N, chunk_size * vocab)
        return out.view(B, N * self.chunk_size, self.vocab_size)  # (B, seq_len, vocab)

# -------------------------
# Hierarchical model (with ablation branches active)
# -------------------------
class HierarchicalCharTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, block_size, ch, quantized_dim, quantization_levels,
                 use_compressor=True, use_float_interp=True, use_mlp_upsampler=True):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.ch = ch
        self.quantized_dim = quantized_dim
        self.quantization_levels = quantization_levels
        self.use_compressor = use_compressor
        self.use_float_interp = use_float_interp
        self.use_mlp_upsampler = use_mlp_upsampler

        assert block_size % ch == 0, "block_size must be divisible by chunk_size"
        assert d_model % quantized_dim == 0, "d_model must be divisible by quantized_dim"
        self.num_chunks = block_size // ch

        if use_compressor:
            self.int_e = ChunkCompressorEmbedding(vocab_size, embedding_dim=32, chunk_size=ch, quantized_dim=quantized_dim)
        else:
            # fallback token embedding (produce chunk embeddings directly)
            self.token_embed = nn.Embedding(vocab_size, d_model)
            self.fallback_proj = nn.Linear(self.ch * self.d_model, self.d_model)

        if use_float_interp:
            float_embed_dim = d_model // quantized_dim
            self.embed = InterpolatedFloatEmbedding(input_dim=quantization_levels, embedding_dim=float_embed_dim)
            self.scale_to_range = nn.Linear(quantized_dim, quantized_dim)
        else:
            # direct linear projection from quantized vectors -> d_model
            self.direct_proj = nn.Linear(quantized_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=4*d_model, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        if use_mlp_upsampler:
            self.upsampler = ChunkReconstructorMLP(d_model, chunk_size=ch, vocab_size=vocab_size)
        else:
            # simpler linear head applied per-chunk, then reshape to token-level logits
            self.direct_head = nn.Linear(d_model, vocab_size)

    def _causal_mask(self, n, device):
        return torch.triu(torch.full((n, n), float('-inf')), diagonal=1).to(device)

    def forward(self, idx, targets=None):
        B, S = idx.shape
        device = idx.device
        assert S == self.block_size, f"Input length {S} must equal model.block_size {self.block_size}"
        num_chunks = S // self.ch

        if self.use_compressor:
            x_chunks = idx.view(B, num_chunks, self.ch)
            x_flat = x_chunks.reshape(-1, self.ch)            # (B*num_chunks, ch)
            quantized_vectors = self.int_e(x_flat)            # (B*num_chunks, quantized_dim)

            if self.use_float_interp:
                scaled = self.scale_to_range(quantized_vectors)      # (B*num_chunks, quantized_dim)
                K = float(self.quantization_levels - 1)
                scaled = (torch.tanh(scaled) + 1.0) / 2.0 * K        # (B*num_chunks, quantized_dim)
                scaled_chunks = scaled.view(B, num_chunks, self.quantized_dim)  # (B, num_chunks, quantized_dim)

                slices = []
                for ch_i in range(self.quantized_dim):
                    channel_values = scaled_chunks[:, :, ch_i]         # (B, num_chunks)
                    emb = self.embed(channel_values)                   # (B, num_chunks, float_embed_dim)
                    slices.append(emb)
                chunk_embs = torch.cat(slices, dim=-1)                # (B, num_chunks, d_model)
            else:
                # direct project quantized -> d_model
                chunk_embs = self.direct_proj(quantized_vectors.view(B, num_chunks, -1))  # (B, num_chunks, d_model)

        else:
            # skip compressor: embed tokens and reshape to chunk-level embeddings
            token_emb = self.token_embed(idx)                          # (B, S, d_model)
            # chunk_embs = token_emb.view(B, num_chunks, self.d_model)   # (B, num_chunks, d_model''
            # token_emb = self.token_embed(idx)  # (B, S, d_model)
            # Reshape to (B, num_chunks, ch, d_model)
            chunked_token_emb = token_emb.view(B, num_chunks, self.ch, self.d_model)
            # Flatten the chunk dimensions and project back to d_model
            flattened_chunks = chunked_token_emb.view(B, num_chunks, self.ch * self.d_model)
            chunk_embs = self.fallback_proj(flattened_chunks) # (B, num_chunks, d_model)

        # transformer encoder across chunks with causal mask
        mask = self._causal_mask(num_chunks, device)
        transformer_out = self.transformer(chunk_embs, mask=mask)      # (B, num_chunks, d_model)

        # upsample back to token logits
        if self.use_mlp_upsampler:
            logits = self.upsampler(transformer_out)                   # (B, S, vocab)
        else:
            # direct_head applied per chunk; then repeat per chunk->token mapping
            # direct_head gives (B, num_chunks, vocab); we expand each chunk into chunk_size tokens
            head = self.direct_head(transformer_out)                   # (B, num_chunks, vocab)
            # expand: repeat each chunk's logits chunk_size times along token axis
            logits = head.unsqueeze(2).repeat(1, 1, self.ch, 1)       # (B, num_chunks, ch, vocab)
            logits = logits.view(B, num_chunks * self.ch, self.vocab_size)  # (B, S, vocab)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# -------------------------
# GENERATION: pad-left sampling (device safe)
# -------------------------
def generate(model, tokenizer, prompt, max_new_tokens, block_size, device, temperature=1.0):
    model.eval()
    model.to(device)
    tokens = tokenizer.encode(prompt)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = tokens[-block_size:]
            context_len = len(context)
            padded = [0] * (block_size - context_len) + context
            idx_input = torch.tensor(padded, dtype=torch.long, device=device).unsqueeze(0)  # (1, block_size)
            logits, _ = model(idx_input)
            last_pos = context_len - 1
            last_logits = logits[:, last_pos, :] / max(1e-8, temperature)
            probs = F.softmax(last_logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            tokens.append(next_idx)
    return tokenizer.decode(tokens)

# -------------------------
# BENCHMARK & TRAINING LOOP
# -------------------------
def benchmark(model, val_loader, device):
    model.eval()
    start = time.time()
    tot_loss = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            _, loss = model(xb, yb)
            tot_loss += loss.item()
            n += 1
    elapsed = time.time() - start
    avg_loss = tot_loss / max(1, n)
    # tokens processed = batches * batch_size * seq_len
    # approximate tokens/sec (if val_loader.batch_size available)
    example_batch = next(iter(val_loader))
    batch_sz = example_batch[0].size(0)
    seq_len = example_batch[0].size(1)
    tokens_processed = n * batch_sz * seq_len
    tps = tokens_processed / elapsed if elapsed > 0 else 0.0
    return {"val_loss": avg_loss, "tokens_per_sec": tps, "elapsed": elapsed}

def train_and_eval(model, train_loader, val_loader, optimizer, device, epochs, tokenizer, save_path=None):
    model.to(device)
    best_val = float('inf')
    for epoch in range(1, epochs+1):
        model.train()
        tot = 0.0
        steps = 0
        start_epoch = time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            _, loss = model(xb, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tot += loss.item()
            steps += 1
        avg_train = tot / max(1, steps)
        val_stats = benchmark(model, val_loader, device)
        avg_val = val_stats["val_loss"]
        tps = val_stats["tokens_per_sec"]

        sample = generate(model, tokenizer, prompt="ROMEO:", max_new_tokens=120, block_size=model.block_size, device=device, temperature=0.8)

        wandb.log({
            "epoch": epoch,
            "train_loss": avg_train,
            "val_loss": avg_val,
            "val_ppl": float(torch.exp(torch.tensor(avg_val))),
            "tokens_per_sec": tps,
            "sample_text": wandb.Html(f"<pre>{sample}</pre>")
        })
        print(f"Epoch {epoch} | train {avg_train:.4f} | val {avg_val:.4f} | ppl {torch.exp(torch.tensor(avg_val)):.3f} | tps {tps:.1f}")

        if avg_val < best_val:
            best_val = avg_val
            if save_path:
                torch.save(model.state_dict(), save_path)
    return best_val

# -------------------------
# SWEEP RUNNER (single-run helper + sweep wrapper)
# -------------------------
def run_single(config):
    # config: a dict-like config (wandb-style)
    text = get_shakespeare_data()
    tokenizer = CharTokenizer(text)
    ids = tokenizer.encode(text)
    split = int(0.9 * len(ids))
    train_ids, val_ids = ids[:split], ids[split:]
    train_ds = CharDataset(train_ids, config["block_size"])
    val_ds   = CharDataset(val_ids, config["block_size"])
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, drop_last=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = HierarchicalCharTransformer(
        vocab_size=tokenizer.vocab_size,
        d_model=config["d_model"],
        nhead=config["nhead"],
        num_layers=config["num_layers"],
        block_size=config["block_size"],
        ch=config["chunk_size"],
        quantized_dim=config["quantized_dim"],
        quantization_levels=config["quant_levels"],
        use_compressor=config["use_compressor"],
        use_float_interp=config["use_float_interp"],
        use_mlp_upsampler=config["use_mlp_upsampler"]
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config.get("weight_decay", 0.0))

    # optional save path
    save_path = f"best.{int(time.time())}.pt"
    best = train_and_eval(model, train_loader, val_loader, optimizer, device, config["epochs"], tokenizer, save_path=save_path)

    final_bench = benchmark(model, val_loader, device)
    final_sample = generate(model, tokenizer, prompt="ROMEO:", max_new_tokens=200, block_size=config["block_size"], device=device, temperature=0.8)

    print("Final val loss:", final_bench["val_loss"])
    print("Final tokens/sec:", final_bench["tokens_per_sec"])
    print("Final sample:\n", final_sample[:1000])
    return save_path, best

def sweep_train(wandb_config=None):
    with wandb.init(config=wandb_config):
        cfg = wandb.config
        config = {
            "epochs": int(cfg.epochs),
            "batch_size": int(cfg.batch_size),
            "block_size": int(cfg.block_size),
            "d_model": int(cfg.d_model),
            "nhead": int(cfg.nhead),
            "num_layers": int(cfg.num_layers),
            "chunk_size": int(cfg.chunk_size),
            "quantized_dim": int(cfg.quantized_dim),
            "quant_levels": int(cfg.quant_levels),
            "lr": float(cfg.lr),
            "weight_decay": float(cfg.weight_decay),
            "use_compressor": bool(cfg.use_compressor),
            "use_float_interp": bool(cfg.use_float_interp),
            "use_mlp_upsampler": bool(cfg.use_mlp_upsampler)
        }
        run_single(config)

# -------------------------
# Sweep config (expanded)
# -------------------------
sweep_config = {
    "method": "random",
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "epochs": {"values": [8]},    # small for fast sweep/dev
        "batch_size": {"values": [16, 32]},
        "block_size": {"values": [32]},
        "d_model": {"values": [128, 192, 256]},
        "nhead": {"values": [4, 8]},
        "num_layers": {"values": [2, 4]},
        "chunk_size": {"values": [2, 4]},
        "quantized_dim": {"values": [4, 8, 16]},
        "quant_levels": {"values": [32, 64, 128]},
        "lr": {"values": [1e-3, 3e-4, 1e-4]},
        "weight_decay": {"values": [0.0, 1e-2]},
        "use_compressor": {"values": [True, False]},
        "use_float_interp": {"values": [True, False]},
        "use_mlp_upsampler": {"values": [True, False]}
    }
}

# -------------------------
# Example: run single config locally (without sweep)
# -------------------------
# Before running: wandb.login()

# config = {
#   "epochs": 6, "batch_size": 32, "block_size": 32,
#   "d_model": 128, "nhead": 4, "num_layers": 4,
#   "chunk_size": 4, "quantized_dim": 8, "quant_levels": 64,
#   "lr": 3e-4, "weight_decay": 0.0,
#   "use_compressor": True, "use_float_interp": True, "use_mlp_upsampler": True
# }
# wandb.init(project="hierarchical-char-transformer", config=config)
# sweep_train(wandb_config=wandb.config)

# -------------------------
# How to run a sweep (uncomment & run)
# -------------------------
# wandb.login()
# sweep_id = wandb.sweep(sweep_config, project="hierarchical-char-transformer")
# wandb.agent(sweep_id, function=sweep_train, count=20)  # run N agents / trials


In [ ]:
import os
wandb.login(key=os.environ.get("WANDB_API_KEY", ""))

In [ ]:
config = {
  "epochs": 6, "batch_size": 32, "block_size": 32,
  "d_model": 128, "nhead": 4, "num_layers": 4,
  "chunk_size": 4, "quantized_dim": 8, "quant_levels": 64,
  "lr": 3e-4, "weight_decay": 0.0,
  "use_compressor": True, "use_float_interp": True, "use_mlp_upsampler": True
}
wandb.init(project="hierarchical-char-transformer", config=config)
sweep_id = wandb.sweep(sweep_config, project="hierarchical-char-transformer")
print(f"Agent runnung on {sweep_id}")
wandb.agent(sweep_id, function=sweep_train, count=20)  # run N agents / trials